In [1]:
import pandas as pd

# File path
file_path = "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/Broad/APOLLO_full_cohort_transformed.csv"

# Read file
df = pd.read_csv(file_path)

# Print shape
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

# Print headers
print("\nHeaders:")
print(df.columns.tolist())

# Optional: print first 5 rows for sanity check
print("\nFirst 5 rows:")
print(df.head())

Number of rows: 1049
Number of columns: 12

Headers:
['REDCap_ID', 'BIOS_ID', 'TP #', 'Blood draw method', 'Tissue progression', 'Progression date', 'Sample Draw date', 'Rapid Autopsy status', 'ER/PR status', 'Her2 Status', 'Histology', 'Unnamed: 11']

First 5 rows:
  REDCap_ID  BIOS_ID       TP # Blood draw method Tissue progression  \
0      4543    57420   TP19-M82        peripheral            ascites   
1      4543    57420  TP21-M297        peripheral            omentum   
2      4543    57420  TP22-M134            periph               bone   
3      4543    57420   TP23-M40      10:00/periph               bone   
4      4543    57420   TP25-M10      10:15/periph            omentum   

  Progression date Sample Draw date Rapid Autopsy status ER/PR status  \
0        1/23/2019        1/24/2019                RA#46     Positive   
1        4/12/2021         5/5/2021                RA#46     Positive   
2         3/9/2022        3/16/2022                RA#46     Positive   
3       

In [3]:
import pandas as pd

file_path = "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/Broad/APOLLO_full_cohort_transformed.csv"
df = pd.read_csv(file_path)

# Clean column names just in case
df.columns = df.columns.str.strip()

# Unique BIOS_ID count
unique_bios = df["BIOS_ID"].nunique()

# Unique Histology values
unique_histology = df["Histology"].dropna().astype(str).str.strip().unique()

# Count each Histology category
histology_counts = df["Histology"].dropna().astype(str).str.strip().value_counts()

# Specific counts
nst_count = histology_counts.get("NST", 0)
ilc_count = histology_counts.get("ILC", 0)

print("Total unique BIOS_ID:", unique_bios)
print("\nTotal unique Histology categories:", len(unique_histology))
print("Unique Histology values:", list(unique_histology))

print("\nHistology counts:")
print(histology_counts)


Total unique BIOS_ID: 345

Total unique Histology categories: 7
Unique Histology values: ['ILC', 'NST', 'Other (other)', 'Other (metaplastic)', 'Other (unknown)', 'Other (mixed)', 'Other (de novo)']

Histology counts:
NST                    878
ILC                    145
Other (metaplastic)     10
Other (unknown)          7
Other (other)            4
Other (mixed)            3
Other (de novo)          2
Name: Histology, dtype: int64


In [4]:
import pandas as pd

file_path = "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/Broad/APOLLO_full_cohort_transformed.csv"
df = pd.read_csv(file_path)

# Clean columns
df.columns = df.columns.str.strip()
df["Histology"] = df["Histology"].astype(str).str.strip()
df["BIOS_ID"] = df["BIOS_ID"].astype(str).str.strip()

# Keep one Histology assignment per BIOS_ID
bios_histology = (
    df[["BIOS_ID", "Histology"]]
    .dropna()
    .drop_duplicates()
)

# Count unique BIOS_ID per Histology
histology_per_bios = (
    bios_histology.groupby("Histology")["BIOS_ID"]
    .nunique()
    .sort_values(ascending=False)
)

print("Total unique BIOS_ID:", df["BIOS_ID"].nunique())
print("\nUnique BIOS_ID by Histology:")
print(histology_per_bios)

Total unique BIOS_ID: 345

Unique BIOS_ID by Histology:
Histology
NST                    281
ILC                     53
Other (unknown)          4
Other (metaplastic)      3
Other (de novo)          2
Other (mixed)            1
Other (other)            1
Name: BIOS_ID, dtype: int64


In [15]:
import pandas as pd

file_path = "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/Broad/APOLLO_full_cohort_transformed.csv"
df = pd.read_csv(file_path)

# Clean columns
df.columns = df.columns.str.strip()
df["BIOS_ID"] = df["BIOS_ID"].astype(str).str.strip()
df["TP #"] = df["TP #"].astype(str).str.strip()
df["Histology"] = df["Histology"].astype(str).str.strip()

# Remove missing TP values
tmp = df.loc[df["TP #"].notna() & (df["TP #"] != ""), ["BIOS_ID", "TP #"]].drop_duplicates()

# Count distinct TP per BIOS_ID
tp_per_bios = tmp.groupby("BIOS_ID")["TP #"].nunique().sort_values(ascending=False)

# Keep BIOS_ID with 3 or more TP
bios_3plus = tp_per_bios[tp_per_bios >= 3].reset_index()
bios_3plus.columns = ["BIOS_ID", "n_TP"]

# Get histology per BIOS_ID
histology_per_bios = (
    df[["BIOS_ID", "Histology"]]
    .drop_duplicates()
    .groupby("BIOS_ID")["Histology"]
    .agg(lambda x: "; ".join(sorted(set(i for i in x if pd.notna(i) and i != "" and i != "nan"))))
    .reset_index()
)

# Merge
bios_3plus_with_histology = bios_3plus.merge(histology_per_bios, on="BIOS_ID", how="left")

# Summary of histology
histology_summary = bios_3plus_with_histology["Histology"].value_counts(dropna=False)

# Print detailed table
print("Number of unique BIOS_ID with 3 or more TP #:", bios_3plus_with_histology.shape[0])
print("\nSummary of Histology among BIOS_ID with 3+ TP #:")
print(histology_summary.to_string())
print("\nBIOS_ID with 3+ TP # and Histology:")
print(bios_3plus_with_histology.to_string(index=False))

Number of unique BIOS_ID with 3 or more TP #: 156

Summary of Histology among BIOS_ID with 3+ TP #:
NST                    129
ILC                     21
Other (metaplastic)      3
Other (other)            1
Other (unknown)          1
Other (mixed)            1

BIOS_ID with 3+ TP # and Histology:
BIOS_ID  n_TP           Histology
  57920    13                 NST
  43480    13                 NST
  70929    13                 NST
  30210    11                 NST
  56754    11                 NST
   3333    10                 NST
  70555    10                 NST
  43280    10                 NST
  42382    10                 NST
  45629    10                 NST
  61527     9                 NST
  42723     9                 NST
  77542     9                 NST
  60938     9                 NST
  68736     9                 NST
  78822     8                 NST
  57420     8                 ILC
  76654     8                 NST
  85443     8                 ILC
  79484     8        

In [16]:
import pandas as pd

file_path = "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/Broad/APOLLO_full_cohort_transformed.csv"
df = pd.read_csv(file_path)

# Clean columns
df.columns = df.columns.str.strip()
df["BIOS_ID"] = df["BIOS_ID"].astype(str).str.strip()
df["TP #"] = df["TP #"].astype(str).str.strip()
df["Histology"] = df["Histology"].astype(str).str.strip()

# Remove missing TP values
tmp = df.loc[df["TP #"].notna() & (df["TP #"] != ""), ["BIOS_ID", "TP #"]].drop_duplicates()

# Count distinct TP per BIOS_ID
tp_per_bios = tmp.groupby("BIOS_ID")["TP #"].nunique().sort_values(ascending=False)

# Keep BIOS_ID with 4 or more TP
bios_4plus = tp_per_bios[tp_per_bios >= 4].reset_index()
bios_4plus.columns = ["BIOS_ID", "n_TP"]

# Get histology per BIOS_ID
histology_per_bios = (
    df[["BIOS_ID", "Histology"]]
    .drop_duplicates()
    .groupby("BIOS_ID")["Histology"]
    .agg(lambda x: "; ".join(sorted(set(i for i in x if pd.notna(i) and i != "" and i != "nan"))))
    .reset_index()
)

# Merge
bios_4plus_with_histology = bios_4plus.merge(histology_per_bios, on="BIOS_ID", how="left")

# Summary of histology
histology_summary = bios_4plus_with_histology["Histology"].value_counts(dropna=False)

# Print detailed table
print("Number of unique BIOS_ID with 4 or more TP #:", bios_4plus_with_histology.shape[0])
print("\nSummary of Histology among BIOS_ID with 4+ TP #:")
print(histology_summary.to_string())
print("\nBIOS_ID with 4+ TP # and Histology:")
print(bios_4plus_with_histology.to_string(index=False))

Number of unique BIOS_ID with 4 or more TP #: 108

Summary of Histology among BIOS_ID with 4+ TP #:
NST                    89
ILC                    17
Other (other)           1
Other (metaplastic)     1

BIOS_ID with 4+ TP # and Histology:
BIOS_ID  n_TP           Histology
  57920    13                 NST
  43480    13                 NST
  70929    13                 NST
  30210    11                 NST
  56754    11                 NST
   3333    10                 NST
  70555    10                 NST
  43280    10                 NST
  42382    10                 NST
  45629    10                 NST
  61527     9                 NST
  42723     9                 NST
  77542     9                 NST
  60938     9                 NST
  68736     9                 NST
  78822     8                 NST
  57420     8                 ILC
  76654     8                 NST
  85443     8                 ILC
  79484     8                 NST
  68677     8                 NST
  56265    